In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
from deep_translator import GoogleTranslator
from deep_translator.constants import GOOGLE_LANGUAGES_TO_CODES
from gtts import gTTS
from playsound import playsound
import os
from PIL import Image, ImageTk
import io
import requests

class SearchableCombobox(ttk.Combobox):
    def __init__(self, master, values, **kwargs):
        super().__init__(master, **kwargs)
        self._values = values
        self['values'] = values
        self.bind('<KeyRelease>', self._filter_values)
        
    def _filter_values(self, event):
        pattern = self.get().lower()
        if not pattern:
            self['values'] = self._values
            return
        filtered = [v for v in self._values if pattern in v.lower()]
        self['values'] = filtered

class TranslatorApp:
    def __init__(self, root):
        self.root = root
        self.setup_ui()
        self.setup_menu()
        
    def setup_ui(self):
        self.root.title("🌍 Advanced Translator Pro")
        self.root.geometry("1000x750")
        self.root.minsize(900, 650)
        self.root.configure(bg="#e6f2ff")

        self.style = ttk.Style()
        self.style.theme_use('clam')
        self.style.configure('.', background="#e6f2ff", foreground="#333333")
        self.style.configure('TFrame', background="#cce0ff")
        self.style.configure('TLabel', background="#cce0ff", foreground="#333333", font=('Helvetica', 10))
        self.style.configure('TButton', background="#4da6ff", foreground="white", 
                            font=('Helvetica', 10, 'bold'), borderwidth=0)
        self.style.map('TButton', background=[('active', '#3385ff')])
        self.style.configure('Header.TLabel', font=('Helvetica', 16, 'bold'), 
                            background="#e6f2ff", foreground="#0066cc")
        self.style.configure('TCombobox', fieldbackground="white", foreground="#333333")
        self.style.configure('TLabelFrame', background="#cce0ff", foreground="#0066cc", 
                           font=('Helvetica', 11, 'bold'))
        self.style.configure('TText', background="white", foreground="#333333", 
                           insertbackground="#333333", font=('Helvetica', 11))
        self.style.configure('TScrollbar', background="#cce0ff")
        self.style.configure('Vertical.TScrollbar', arrowcolor="#0066cc")
        self.style.configure('Horizontal.TScrollbar', arrowcolor="#0066cc")

        self.main_frame = ttk.Frame(self.root)
        self.main_frame.pack(fill=tk.BOTH, expand=True, padx=20, pady=20)
        
        self.header_frame = ttk.Frame(self.main_frame)
        self.header_frame.pack(fill=tk.X, pady=(0, 20))
        
        try:
            response = requests.get("https://cdn-icons-png.flaticon.com/512/3898/3898082.png")
            image = Image.open(io.BytesIO(response.content))
            image = image.resize((50, 50), Image.LANCZOS)
            self.logo = ImageTk.PhotoImage(image)
            ttk.Label(self.header_frame, image=self.logo, background="#e6f2ff").pack(side=tk.LEFT, padx=(0, 15))
        except:
            pass
            
        ttk.Label(self.header_frame, text="Advanced Translator Pro", style='Header.TLabel').pack(side=tk.LEFT)
        
        self.input_frame = ttk.LabelFrame(self.main_frame, text="Source Text", padding=15)
        self.input_frame.pack(fill=tk.BOTH, expand=True)
        
        self.input_scroll = ttk.Scrollbar(self.input_frame)
        self.input_scroll.pack(side=tk.RIGHT, fill=tk.Y)
        
        self.input_text = tk.Text(self.input_frame, height=10, wrap=tk.WORD, 
                                yscrollcommand=self.input_scroll.set)
        self.input_text.pack(fill=tk.BOTH, expand=True)
        self.input_scroll.config(command=self.input_text.yview)
        
        self.control_frame = ttk.Frame(self.main_frame)
        self.control_frame.pack(fill=tk.X, pady=15)
        
        self.langs = [f"{code} - {lang.title()}" for lang, code in GOOGLE_LANGUAGES_TO_CODES.items()]
        self.langs.sort()
        
        self.source_lang = tk.StringVar(value="Select Language")
        self.target_lang = tk.StringVar(value="Select Language")
        
        ttk.Label(self.control_frame, text="From:").grid(row=0, column=0, padx=(0, 5), sticky=tk.W)
        self.source_combo = SearchableCombobox(
            self.control_frame, 
            textvariable=self.source_lang, 
            values=["Select Language"] + self.langs, 
            width=30,
            state="normal"
        )
        self.source_combo.grid(row=0, column=1, padx=5, sticky=tk.W)
        
        self.swap_btn = ttk.Button(
            self.control_frame, 
            text="⇄ Swap", 
            width=8, 
            command=self.swap_languages
        )
        self.swap_btn.grid(row=0, column=2, padx=10)
        
        ttk.Label(self.control_frame, text="To:").grid(row=0, column=3, padx=(20, 5), sticky=tk.W)
        self.target_combo = SearchableCombobox(
            self.control_frame, 
            textvariable=self.target_lang, 
            values=self.langs, 
            width=30,
            state="normal"
        )
        self.target_combo.grid(row=0, column=4, padx=5, sticky=tk.W)
        
        self.btn_frame = ttk.Frame(self.control_frame)
        self.btn_frame.grid(row=0, column=5, padx=(20, 0))
        
        self.translate_btn = ttk.Button(
            self.btn_frame, 
            text="Translate", 
            command=self.translate_text
        )
        self.translate_btn.pack(side=tk.LEFT, padx=5)
        
        self.clear_btn = ttk.Button(
            self.btn_frame, 
            text="Clear", 
            command=self.clear_text
        )
        self.clear_btn.pack(side=tk.LEFT, padx=5)
        
        self.output_frame = ttk.LabelFrame(self.main_frame, text="Translation", padding=15)
        self.output_frame.pack(fill=tk.BOTH, expand=True)
        
        self.output_scroll = ttk.Scrollbar(self.output_frame)
        self.output_scroll.pack(side=tk.RIGHT, fill=tk.Y)
        
        self.output_text = tk.Text(self.output_frame, height=10, wrap=tk.WORD, 
                                 yscrollcommand=self.output_scroll.set)
        self.output_text.pack(fill=tk.BOTH, expand=True)
        self.output_scroll.config(command=self.output_text.yview)
        
        self.output_btn_frame = ttk.Frame(self.main_frame)
        self.output_btn_frame.pack(fill=tk.X, pady=(15, 0))
        
        self.copy_btn = ttk.Button(
            self.output_btn_frame, 
            text="Copy Translation", 
            command=self.copy_text
        )
        self.copy_btn.pack(side=tk.LEFT, padx=5)
        
        self.speak_btn = ttk.Button(
            self.output_btn_frame, 
            text="Speak Translation", 
            command=self.speak_text
        )
        self.speak_btn.pack(side=tk.LEFT, padx=5)
        
        self.save_btn = ttk.Button(
            self.output_btn_frame, 
            text="Save to File", 
            command=self.save_to_file
        )
        self.save_btn.pack(side=tk.LEFT, padx=5)
        
        self.status_var = tk.StringVar()
        self.status_var.set("Ready")
        self.status_bar = ttk.Label(
            self.main_frame, 
            textvariable=self.status_var, 
            relief=tk.SUNKEN, 
            anchor=tk.W,
            padding=5
        )
        self.status_bar.pack(fill=tk.X, pady=(20, 0))
        
        self.control_frame.columnconfigure(5, weight=1)
        
    def setup_menu(self):
        self.menubar = tk.Menu(self.root, bg="#cce0ff", fg="#333333")
        
        self.file_menu = tk.Menu(self.menubar, tearoff=0, bg="#cce0ff", fg="#333333")
        self.file_menu.add_command(label="Open Text File", command=self.open_file)
        self.file_menu.add_command(label="Save Translation", command=self.save_to_file)
        self.file_menu.add_separator()
        self.file_menu.add_command(label="Exit", command=self.root.quit)
        self.menubar.add_cascade(label="File", menu=self.file_menu)
        
        self.help_menu = tk.Menu(self.menubar, tearoff=0, bg="#cce0ff", fg="#333333")
        self.help_menu.add_command(label="About", command=self.show_about)
        self.help_menu.add_command(label="Help", command=self.show_help)
        self.menubar.add_cascade(label="Help", menu=self.help_menu)
        
        self.root.config(menu=self.menubar)
    
    def translate_text(self):
        text = self.input_text.get("1.0", tk.END).strip()
        if not text:
            messagebox.showwarning("Input required", "Please enter text to translate.")
            return
        
        self.status_var.set("Translating...")
        self.root.update()
        
        src = self.source_lang.get().split(" - ")[0]
        dest = self.target_lang.get().split(" - ")[0]
        
        try:
            translated = GoogleTranslator(source=src, target=dest).translate(text)
            self.output_text.delete("1.0", tk.END)
            self.output_text.insert(tk.END, translated)
            self.status_var.set(f"Translated from {src} to {dest}")
        except Exception as e:
            messagebox.showerror("Translation Error", str(e))
            self.status_var.set("Error in translation")
    
    def copy_text(self):
        text = self.output_text.get("1.0", tk.END).strip()
        if text:
            self.root.clipboard_clear()
            self.root.clipboard_append(text)
            self.status_var.set("Translation copied to clipboard")
    
    def speak_text(self):
        text = self.output_text.get("1.0", tk.END).strip()
        if not text:
            return
            
        self.status_var.set("Generating speech...")
        self.root.update()
        
        lang_code = self.target_lang.get().split(" - ")[0]
        try:
            tts = gTTS(text=text, lang=lang_code)
            filename = "temp_audio.mp3"
            tts.save(filename)
            playsound(filename)
            os.remove(filename)
            self.status_var.set("Speech completed")
        except Exception as e:
            messagebox.showerror("TTS Error", str(e))
            self.status_var.set("Error in speech generation")
    
    def swap_languages(self):
        current_source = self.source_lang.get()
        current_target = self.target_lang.get()
        
        if current_source == "Select Language":
            return
            
        self.source_lang.set(current_target)
        self.target_lang.set(current_source)
        
        input_text = self.input_text.get("1.0", tk.END).strip()
        output_text = self.output_text.get("1.0", tk.END).strip()
        
        if output_text:
            self.input_text.delete("1.0", tk.END)
            self.input_text.insert(tk.END, output_text)
            self.output_text.delete("1.0", tk.END)
            if input_text:
                self.output_text.insert(tk.END, input_text)
    
    def clear_text(self):
        self.input_text.delete("1.0", tk.END)
        self.output_text.delete("1.0", tk.END)
        self.status_var.set("Cleared text")
    
    def save_to_file(self):
        text = self.output_text.get("1.0", tk.END).strip()
        if not text:
            messagebox.showwarning("No Content", "No translation to save.")
            return
            
        filepath = filedialog.asksaveasfilename(
            defaultextension=".txt",
            filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")],
            title="Save Translation"
        )
        
        if filepath:
            try:
                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(text)
                self.status_var.set(f"Translation saved to {os.path.basename(filepath)}")
            except Exception as e:
                messagebox.showerror("Save Error", str(e))
                self.status_var.set("Error saving file")
    
    def open_file(self):
        filepath = filedialog.askopenfilename(
            filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")],
            title="Open Text File"
        )
        
        if filepath:
            try:
                with open(filepath, 'r', encoding='utf-8') as f:
                    content = f.read()
                self.input_text.delete("1.0", tk.END)
                self.input_text.insert(tk.END, content)
                self.status_var.set(f"Loaded {os.path.basename(filepath)}")
            except Exception as e:
                messagebox.showerror("Open Error", str(e))
                self.status_var.set("Error opening file")
    
    def show_about(self):
        about_text = """Advanced Translator Pro v2.0

A powerful translation tool with:
- Real-time language search
- Text-to-speech capability
- Modern light blue interface

Developed with Python using:
- Google Translate API
- gTTS for speech synthesis

© 2025 Translator Pro Team"""
        messagebox.showinfo("About", about_text)
    
    def show_help(self):
        help_text = """How to use the translator:

1. Enter text in the source area or open a text file
2. Select source and target languages (type to search)
3. Click Translate button
4. Use the output buttons to copy, speak, or save the translation

Tips:
- Use the swap button (⇄) to quickly switch languages
- Auto Detect will try to identify the source language
- Type in the language dropdown to search for languages"""
        messagebox.showinfo("Help", help_text)

if __name__ == "__main__":
    root = tk.Tk()
    app = TranslatorApp(root)
    root.mainloop()